# Basic Perseus CTS Workflow

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Introduction</a>
* <a href="#cts-and-urns">2 - CTS requests and URN structure</a>
* <a href="#setup">3 - Setup and request helper</a>
* <a href="#fetch-passage">4 - Fetch and parse a passage</a>
* <a href="#valid-references">5 - Inspect valid references</a>
* <a href="#label-metadata">6 - Retrieve label metadata</a>
* <a href="#mcp-mapping">7 - From raw CTS to the MCP tools</a>
* <a href="#mcp-benefits">8 - What the MCP layer adds</a>
* <a href="#next-steps">9 - Continue learning</a>
* <a href="#sources">10 - Sources</a>
* <a href="#required-libraries">11 - Required libraries</a>
* <a href="#notebook-version">12 - Notebook version</a>

## 1 - Introduction <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook introduces the **Canonical Text Services (CTS)** interface used underneath the Perseus MCP server. It deliberately calls the upstream Perseus endpoint directly, without FastMCP, so that the requests and XML responses remain visible.

By the end of the notebook you will be able to:

- recognize the main parts of a CTS URN;
- send a reusable CTS request with `httpx`;
- retrieve a passage and turn the XML into readable text;
- inspect valid citation references and identify the resolved edition;
- retrieve human-readable metadata for a CTS resource;
- understand how these raw operations relate to the project's 23 MCP tools.

> This is a protocol-learning notebook. For normal application or agent use, prefer the MCP tools: they add discovery helpers, structured JSON, plaintext conversion, caching, search, and navigation fallbacks.

## 2 - CTS requests and URN structure <a class="anchor" id="cts-and-urns"></a>
##### [Back to ToC](#TOC)

CTS identifies texts and passages with hierarchical **URNs** (Uniform Resource Names). The examples below move from a broad author/textgroup identifier to a specific passage:

| Scope | Example | Meaning |
|---|---|---|
| Textgroup | `urn:cts:greekLit:tlg0012` | Homer |
| Work | `urn:cts:greekLit:tlg0012.tlg001` | Homer's *Iliad* |
| Edition | `urn:cts:greekLit:tlg0012.tlg001.perseus-grc1` | One advertised Greek edition |
| Passage | `urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1-1.5` | Lines 1.1 through 1.5 in that edition |

The part before the colon identifies the textual resource; the part after the colon is the citation. A work-level passage such as `urn:cts:greekLit:tlg0012.tlg001:1.1-1.5` asks Perseus to resolve a suitable edition. That is convenient for exploration, but reproducible workflows should normally discover and record an edition-specific URN.

The live CTS inventory can change, and Perseus CTS and Scaife may advertise different edition identifiers. MCP workflows should therefore discover resources with `list_text_groups`, `get_author_resources`, or `get_work_resources` before constructing edition-specific passage URNs.

## 3 - Setup and request helper <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

This notebook only needs `httpx`; XML and JSON support come from Python's standard library. We use `%pip` because it installs into the active Jupyter kernel.

The `cts_request` helper centralizes the endpoint, timeout, status checking, and optional URN. Keeping those details in one function makes the individual examples easier to read.

In [ ]:
%pip install --quiet httpx

In [ ]:
import json
import xml.etree.ElementTree as ET

import httpx

CTS_BASE = "https://www.perseus.tufts.edu/hopper/CTS"
HOMER_TEXTGROUP = "urn:cts:greekLit:tlg0012"
ILIAD_WORK = "urn:cts:greekLit:tlg0012.tlg001"
PASSAGE_URN = f"{ILIAD_WORK}:1.1-1.5"


def cts_request(request, urn=None, timeout=20.0, **extra_params):
    """Send one request to the Perseus CTS endpoint and return its text."""
    params = {"request": request, **extra_params}
    if urn is not None:
        params["urn"] = urn

    response = httpx.get(
        CTS_BASE,
        params=params,
        timeout=timeout,
        follow_redirects=True,
    )
    response.raise_for_status()
    return response.text


def local_name(tag):
    """Return an XML tag without its namespace."""
    return tag.rsplit("}", 1)[-1]

## 4 - Fetch and parse a passage <a class="anchor" id="fetch-passage"></a>
##### [Back to ToC](#TOC)

`GetPassage` returns XML containing request metadata and TEI-encoded text. First inspect a short prefix of the raw response. Rerun this cell when working with the notebook: the output is intentionally not stored because it comes from a live service.

In [ ]:
passage_xml = cts_request("GetPassage", urn=PASSAGE_URN)
print(passage_xml[:1200])

### 4.1 - Convert the XML to readable text

XML namespaces make raw tag names look like `{namespace}tag`. The `local_name` helper removes that prefix. The extraction function below prefers common TEI text-bearing elements and falls back to broader containers when necessary.

This is the basic transformation performed for you by the MCP tool `get_passage_plaintext`.

In [ ]:
def element_text(element):
    return " ".join("".join(element.itertext()).split())


def passage_plaintext(xml_text):
    root = ET.fromstring(xml_text)
    text_parts = []

    for element in root.iter():
        if local_name(element.tag) not in {"l", "p", "ab", "seg", "quote"}:
            continue
        text = element_text(element)
        if text and text not in text_parts:
            text_parts.append(text)

    if not text_parts:
        for element in root.iter():
            if local_name(element.tag) in {"text", "body", "div"}:
                text = element_text(element)
                if text:
                    text_parts.append(text)
                    break

    return "\n".join(text_parts)


print(passage_plaintext(passage_xml))

## 5 - Inspect valid references <a class="anchor" id="valid-references"></a>
##### [Back to ToC](#TOC)

`GetValidReff` returns the citations that CTS considers valid beneath a work or edition. This is essential for safe navigation: it is better to select an advertised reference than to guess that a chapter, line, or section exists.

The response may be large, so the example parses it immediately instead of printing all the XML. Notice that a work-level request can resolve to an edition-specific list.

In [ ]:
references_xml = cts_request("GetValidReff", urn=ILIAD_WORK)
references_root = ET.fromstring(references_xml)

reference_urns = [
    element_text(element)
    for element in references_root.iter()
    if local_name(element.tag) == "urn" and element_text(element)
]

resolved_edition = reference_urns[0].rpartition(":")[0] if reference_urns else None
reference_summary = {
    "requested_work": ILIAD_WORK,
    "resolved_edition": resolved_edition,
    "total_count": len(reference_urns),
    "first_page": reference_urns[:10],
    "has_more": len(reference_urns) > 10,
}

print(json.dumps(reference_summary, ensure_ascii=False, indent=2))

### 5.1 - Why the MCP reference helpers matter

The raw CTS response can contain thousands of references. The MCP server caches this stable metadata and offers three levels of access:

- `get_valid_references` returns the original XML;
- `get_valid_references_json` returns a small page with `limit`, `offset`, and `has_next` metadata;
- `count_valid_references` returns only the count.

For agents and user interfaces, the paged JSON or count tools are usually safer than moving the complete XML through every step.

## 6 - Retrieve label metadata <a class="anchor" id="label-metadata"></a>
##### [Back to ToC](#TOC)

`GetLabel` asks CTS for human-readable information associated with a work or edition URN. The exact XML shape can vary, so the code collects useful fields by local tag name rather than relying on one fixed namespace prefix.

In [ ]:
label_xml = cts_request("GetLabel", urn=ILIAD_WORK)
label_root = ET.fromstring(label_xml)

wanted_fields = {"groupname", "title", "label", "description", "versionInfo"}
label_metadata = {}

for element in label_root.iter():
    name = local_name(element.tag)
    text = element_text(element)
    if name in wanted_fields and text:
        label_metadata.setdefault(name, [])
        if text not in label_metadata[name]:
            label_metadata[name].append(text)

print(json.dumps(label_metadata, ensure_ascii=False, indent=2))

## 7 - From raw CTS to the MCP tools <a class="anchor" id="mcp-mapping"></a>
##### [Back to ToC](#TOC)

The MCP server exposes the useful CTS operations while adding more convenient output formats and higher-level workflows.

| Raw CTS operation | Related MCP tools | What changes at the MCP layer |
|---|---|---|
| `GetPassage` | `get_passage`, `get_passage_plaintext` | Choose raw XML or readable text |
| `GetPassagePlus` | `get_passage_plus` | Retrieve passage text with additional CTS context |
| `GetValidReff` | `get_valid_references`, `get_valid_references_json`, `count_valid_references` | Choose raw XML, paged JSON, or a lightweight count |
| `GetCapabilities` | `get_capabilities`, `list_text_groups`, `find_author_names`, `get_author_resources`, `get_work_resources` | Turn a large inventory document into focused discovery operations |
| `GetLabel` | `get_label` | Retrieve labels through a stable tool interface |
| `GetFirstUrn` | `get_first_urn` | Fall back to valid-reference data if the live CTS response is malformed |
| `GetPrevNextUrn` | `get_prev_next_urn` | Fall back to ordered valid references when necessary |

The complete 23-tool surface is organized by research task:

| Task | Tools |
|---|---|
| Passage retrieval | `get_passage`, `get_passage_plus`, `get_passage_plaintext` |
| Citation references | `get_valid_references`, `get_valid_references_json`, `count_valid_references` |
| Inventory and discovery | `get_capabilities`, `list_text_groups`, `find_author_names`, `get_author_resources`, `get_work_resources` |
| Labels and navigation | `get_label`, `get_first_urn`, `get_prev_next_urn` |
| Library and in-text search | `search_perseus`, `search_within_text`, `get_passage_highlights` |
| Scaife-native retrieval | `get_scaife_library_metadata`, `get_scaife_passage_json`, `get_scaife_passage_text` |
| Cache management | `get_cache_status`, `refresh_metadata_cache`, `clear_metadata_cache` |

Notebook `05_mcp_all_tools.ipynb` can inspect the live FastMCP catalog, including each tool's current description and JSON input schema.

## 8 - What the MCP layer adds <a class="anchor" id="mcp-benefits"></a>
##### [Back to ToC](#TOC)

The direct requests above are valuable for understanding the source protocol, but the MCP layer is preferable for most research workflows because it adds:

- **resource discovery** before URN construction, reducing assumptions about currently available editions;
- **readable and structured outputs**, including plaintext passages and paged JSON references;
- **metadata caching** for large, relatively stable `GetCapabilities` and `GetValidReff` responses;
- **navigation fallbacks** for live Perseus responses that are occasionally malformed;
- **Greek search normalization**, including Unicode Greek and common Beta Code input;
- **Scaife search and retrieval**, covering library search, edition-scoped search, token highlights, and Scaife-native passage data;
- **one MCP interface** that can be used from Python, desktop clients, coding agents, or an LLM tool loop.

A useful mental model is: **CTS supplies the canonical text and citation layer; Scaife adds search and complementary retrieval; the MCP server turns both into focused research tools.**

## 9 - Continue learning <a class="anchor" id="next-steps"></a>
##### [Back to ToC](#TOC)

Continue with the notebook that matches your next question:

- [`02_search_and_navigation.ipynb`](02_search_and_navigation.ipynb) — compare direct Scaife search with CTS citation navigation;
- [`03_mcp_connection_homer_iliad.ipynb`](03_mcp_connection_homer_iliad.ipynb) — call the same research operations through an in-process FastMCP client;
- [`04_mcp_greek_search_and_navigation.ipynb`](04_mcp_greek_search_and_navigation.ipynb) — combine Unicode/Beta Code search, discovery, passage retrieval, and navigation;
- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) — inspect every registered tool and its input schema;
- [`08_mcp_new_cache_and_search_tools.ipynb`](08_mcp_new_cache_and_search_tools.ipynb) — explore cache controls, paging, scoped search, highlights, and Scaife-native retrieval.

A good first experiment in this notebook is to change `PASSAGE_URN` to another short work-level citation. For an edition-specific experiment, first obtain the current edition URN from an MCP discovery notebook rather than copying an identifier from an old response.

## 10 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook uses and describes:

- the [Perseus Digital Library](https://www.perseus.tufts.edu/);
- the Perseus [CTS endpoint](https://www.perseus.tufts.edu/hopper/CTS), including `GetPassage`, `GetValidReff`, and `GetLabel`;
- the [Scaife Viewer](https://scaife.perseus.org/) services used by the search and Scaife-native MCP tools;
- the local MCP implementation in [`server.py`](../server.py);
- the project overview and current tool list in the [README](../README.md).

Perseus and Scaife are live upstream services. Available editions, result counts, ordering, and response details may change over time. Treat the code as the reproducible example and rerun it to obtain current data.

## 11 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

This repository targets **Python 3.11 or newer**. The notebook directly requires:

- `httpx>=0.27.0` for HTTP requests;
- Jupyter/IPython to run the cells.

The `json` and `xml.etree.ElementTree` modules are included with Python and need no separate installation. The notebook's `%pip install --quiet httpx` cell installs `httpx` into the active kernel. Alternatively, install all project dependencies from the repository root with:

```bash
pip install -e .
```

## 12 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.2</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>